# HydroSalNet v2 — Kaggle end-to-end training (fixed + restartable)
### Quality-Aware Multimodal Soil EC1:5 Regression | 2024–2025

**Production notebook for the 349 × 548 HydroSalNet master dataset.** This revision fixes errors from earlier copies: no hard-coded Kaggle dataset pathname or filename, reliable initialization, safe write locations, consistent resumption, and protected model/results ledgers.

**Pipeline**
1. Auto-detect the **correct dataset by its content** under `/kaggle/input`, even if Kaggle renamed the CSV or nested its directory. Also accepts a ZIP containing the CSV. **Never silently train on a 349-row but wrong dataset**.
2. Audit ground EC labels, source availability, 2024/2025 campaign differences, missingness, optical quality, correlations and geo-data; save high-resolution bar, pie, line, scatter, box and heatmap visualizations.
3. Select transparent environmental predictors grouped into optical, SAR, soil, climate, hydrology and terrain; **no EC, sample ID, coordinates, dates, year or administrative labels as model predictors**.
4. Freeze **2024-only random CV (diagnostic)**, **2024-only spatial block CV (primary)** and **2024 train → 2025 holdout (cross-campaign)**. Fit missingness preprocessing exclusively inside training folds.
5. Train **DummyMean, Ridge, ElasticNet, RandomForest, ExtraTrees, XGBoost, CatBoost, LightGBM, MLP_NoQA, MLP_EarlyFusion_QA, HydroSalNet_Gated**.
6. Use **50 neural epochs** and **50 trees/boosting rounds** for applicable models; single-fit linear methods do not have epochs. Save weights/optimizer after **every neural epoch** and tree/boosting models in configurable chunks.
7. Compare metrics, generate plots and ablations, save preprocessor + models + results + recovery archive.

> **Important scientific caveat:** 2025 EC laboratory protocol requires source-document reconciliation. March 2024 versus April 2025 mixes geography, month and campaign; it is **not** an isolated annual trend. Results are a pilot benchmark until stricter cross-region and lab QA is complete.

> **Crash recovery caveat:** `/kaggle/working` is writable but not automatically durable across a lost session. For recovery after a failed Kaggle session, enable and **verify** private remote checkpoint sync (optional) or restore from a previously saved notebook-version output. Browser/laptop power loss does not itself stop a cloud **Save & Run All**, but server-side failures can stop it. Never assume that saving files locally guarantees a restart.


## 0. Before running — Kaggle setup

1. Kaggle → **Add Input** → attach the dataset named **HydroSalNet**. The notebook scans recursively for a CSV with `sample_id`, `EC_1to5_mScm`, 349 rows and 548 columns; the actual input filename **does not have to match** the original master filename.
2. The first configuration cell does **not** read the dataset yet; it just creates variables and the writable output directory. **Run All**, not an isolated training cell, from a fresh kernel.
3. Default settings: `EPOCHS=50`, `BOOSTING_ROUNDS=50`, `TREE_COUNT=50`, `FAST_DEV_RUN=False`. A shortened fast test is available for checking your Kaggle environment.
4. For unattended runs that must survive cloud session failure: enable Internet, add Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`, switch `ENABLE_REMOTE_BACKUP=True`, and **confirm the initial private upload succeeds**. Optional uploads contain model files and metrics, *not the raw EC CSV*. Even then the most recent unsynced steps can be lost.
5. Use **Save Version → Save & Run All** to run in a Kaggle-managed background version and preserve completed outputs. If you only run interactively and the session dies, local files might disappear.

**Restart rules:** use the same `RUN_NAME` and complete same config, reattach the source master and most recent `hydrosalnet-recovery.zip` as Kaggle inputs, then choose **Run All**. Changing the model budget, seed, feature list, or folds requires a **different RUN_NAME** to prevent mixing incompatible results.


In [ ]:
# ============================================================
# HydroSalNet v2 — CONFIGURATION: run this cell FIRST
# ============================================================
from pathlib import Path
import os

RUN_NAME = "hydrosalnet_v2_50epoch_20261117"  # change for any new experimental setup
RANDOM_SEED = 42

# Kaggle automatically mounts datasets under /kaggle/input (the subfolder can vary).
INPUT_ROOT = Path(os.environ.get("HYDROSALNET_INPUT_ROOT", "/kaggle/input"))
# Leave blank for automatic CSV/ZIP discovery, even when the CSV has a different filename.
DATA_PATH = os.environ.get("HYDROSALNET_CSV", "").strip()
PREFERRED_CSV_NAME = "HydroSalNet_Final_Master_2024_2025.csv"
EXPECTED_ROWS, EXPECTED_COLUMNS = 349, 548

# Output MUST be writable. Never use /kaggle/input as WORK_DIR.
WORK_DIR = Path(os.environ.get(
    "HYDROSALNET_OUT", str(Path('/kaggle/working/hydrosalnet_training') / RUN_NAME)
))
for part in ["", "eda", "results", "models", "checkpoints", "logs"]:
    (WORK_DIR / part).mkdir(parents=True, exist_ok=True)

CV_FOLDS = 4
EPOCHS = 50                   # neural network epochs
BOOSTING_ROUNDS = 50         # XGBoost / CatBoost / LightGBM rounds
TREE_COUNT = 50              # Random Forest / Extra Trees total estimators
CHECKPOINT_EVERY_TREE_STEP = 10

NN_HIDDEN = 48
NN_EMBED = 12
NN_LR = 0.003
NN_WEIGHT_DECAY = 0.01
NN_BATCH_SIZE = 32
NN_DROPOUT = 0.15
MODALITY_DROPOUT = 0.15

RUN_EDA = True
RUN_ABLATIONS = True
FAST_DEV_RUN = False          # True only for 2-fold/2-epoch/4-tree smoke run

# OPTIONAL real cross-session recovery; requires Internet + Kaggle Secrets/API access.
ENABLE_REMOTE_BACKUP = False
REMOTE_DATASET_SLUG = 'hydrosalnet-private-training-checkpoints-v2'
REMOTE_SYNC_EVERY_N_EPOCHS = 5
REMOTE_MIN_SECONDS_BETWEEN_UPLOADS = 120
RESTORE_ZIP_PATH = ''         # optional, e.g. attached private checkpoint ZIP

if FAST_DEV_RUN:
    CV_FOLDS, EPOCHS, BOOSTING_ROUNDS, TREE_COUNT = 2, 2, 4, 4
    CHECKPOINT_EVERY_TREE_STEP = 2
    RUN_NAME += "_SMOKE"
    WORK_DIR = WORK_DIR / "smoke_run"  # never mix smoke artifacts with the full run
    for part in ["", "eda", "results", "models", "checkpoints", "logs"]:
        (WORK_DIR / part).mkdir(parents=True, exist_ok=True)

if str(WORK_DIR).startswith('/kaggle/input'):
    raise ValueError('WORK_DIR is under read-only /kaggle/input. Use /kaggle/working.')

print('RUN_NAME:',RUN_NAME)
print('Input search root:',INPUT_ROOT)
print('Explicit data override:',DATA_PATH or '[AUTOMATIC DETECTION]')
print('Writable outputs:',WORK_DIR)
print('Epochs / rounds / trees:',EPOCHS, BOOSTING_ROUNDS, TREE_COUNT)
print('Remote backup:',ENABLE_REMOTE_BACKUP,'| FAST_DEV_RUN:',FAST_DEV_RUN)


## 1. Packages and environment checks

Kaggle usually includes the packages below. The dependency cell installs missing optional gradient-boosting libraries only when necessary; Internet must be ON for installs.

In [ ]:
import sys, subprocess, importlib.util
required_packages = {"pandas":"pandas", "numpy":"numpy", "matplotlib":"matplotlib",
                     "sklearn":"scikit-learn", "seaborn":"seaborn", "torch":"torch",
                     "joblib":"joblib", "xgboost":"xgboost", "catboost":"catboost",
                     "lightgbm":"lightgbm"}
missing = [package for module,package in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Environment ready; Python:", sys.version.split()[0])

In [ ]:
import os, sys, csv, json, math, re, io, glob, time, hashlib, zipfile, shutil
import warnings, random, traceback, subprocess
from datetime import datetime, timezone
from collections import OrderedDict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold, GroupKFold
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
import xgboost as xgb
import catboost as cb
import lightgbm as lgb

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for p in [WORK_DIR, WORK_DIR/"eda", WORK_DIR/"results", WORK_DIR/"models", WORK_DIR/"checkpoints", WORK_DIR/"logs"]:
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=0.92)
print("Compute:", DEVICE, "| sklearn:", __import__('sklearn').__version__,
      "| xgboost:", xgb.__version__, "| catboost:", cb.__version__,
      "| lightgbm:", lgb.__version__, "| torch:", torch.__version__)

def savefig(name):
    path = WORK_DIR/"eda"/(name+".png") if not name.startswith("results_") else WORK_DIR/"results"/(name+".png")
    plt.tight_layout(); plt.savefig(path, dpi=165, bbox_inches="tight")
    plt.show(); plt.close("all")
    return path

def atomic_json(path, item):
    path = Path(path); tmp = path.with_suffix(path.suffix+".tmp")
    with open(tmp,"w",encoding="utf-8") as f: json.dump(item,f,indent=2,default=str)
    os.replace(tmp,path)

def save_csv_atomic(data, path):
    path = Path(path); tmp = path.with_suffix(path.suffix+".tmp")
    data.to_csv(tmp,index=False); os.replace(tmp,path)

def save_joblib_atomic(obj,path):
    """Persist models without leaving a partially written destination file."""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.writing')
    joblib.dump(obj,temporary)
    os.replace(temporary,path)


def append_row_unique(file_path,row,keys):
    """Atomic CSV upsert by key; safe to re-run a completed fold after interruption."""
    file_path=Path(file_path)
    new=pd.DataFrame([row])
    prior=pd.read_csv(file_path) if file_path.exists() else pd.DataFrame(columns=new.columns)
    if len(prior):
        mask=pd.Series(True,index=prior.index)
        for k in keys: mask &= prior[k].astype(str).eq(str(row[k]))
        prior=prior.loc[~mask]
    save_csv_atomic(pd.concat([prior,new],ignore_index=True),file_path)


def upsert_prediction_rows(file_path,new_rows):
    file_path=Path(file_path)
    old=pd.read_csv(file_path) if file_path.exists() else pd.DataFrame(columns=new_rows.columns)
    if not old.empty:
        keys=['protocol','fold','model']
        criterion=pd.Series(True,index=old.index)
        for k in keys: criterion &= old[k].eq(str(new_rows.iloc[0][k]))
        old=old.loc[~criterion]
    save_csv_atomic(pd.concat([old,new_rows],ignore_index=True),file_path)


## 2. Detect, audit and fingerprint the **actual attached master**

The user-reported Kaggle dataset root is similar to `/kaggle/input/datasets/syedtamim2022566642/hydrosalnet`, **but that is not a guarantee of its mounted filename or folder**. This notebook inspects the attached input **recursively**. It accepts a renamed CSV (including a Kaggle version suffix), or the final-master CSV inside a ZIP. A candidate must have the required ground EC fields and expected shape. If several different valid masters exist, it **stops and asks you to pick one** rather than silently training on arbitrary data.

When restarting, **attach your own private** `hydrosalnet-recovery.zip` under Kaggle Input, or set `RESTORE_ZIP_PATH`. Checkpoint fingerprints are checked before they can be reused.


In [ ]:
# STEP 2. Dataset discovery + safe restore + full source validation.
# Works regardless of CSV display name or Kaggle's changing mount structure.
import io

REQUIRED_COLUMNS = ['sample_id', 'sample_year', 'sample_date',
                    'latitude', 'longitude', 'EC_1to5_mScm']

def scan_files(root, suffixes):
    root = Path(root)
    if not root.exists(): return []
    return sorted(p for p in root.rglob('*') if p.is_file()
                  and p.suffix.lower() in suffixes)

def csv_header_candidate(path_or_buffer):
    try:
        trial = pd.read_csv(path_or_buffer, nrows=2, low_memory=False)
        trial.columns = trial.columns.astype(str).str.lstrip('\ufeff')
        return all(c in trial.columns for c in REQUIRED_COLUMNS) and len(trial.columns) == EXPECTED_COLUMNS
    except Exception:
        return False

# An explicit path is honored only when valid; else provide actionable diagnostics.
source_candidates = []
if DATA_PATH:
    if not Path(DATA_PATH).is_file():
        raise FileNotFoundError(f"Configured DATA_PATH does not exist: {DATA_PATH}")
    source_candidates.append((Path(DATA_PATH), None))
else:
    csvs = scan_files(INPUT_ROOT, {'.csv'})
    print(f'Found {len(csvs)} CSV file(s) within {INPUT_ROOT}:')
    for p in csvs[:40]: print('  -',p)
    for p in csvs:
        if csv_header_candidate(p): source_candidates.append((p, None))
    if not source_candidates:
        # Can also use the official ZIP containing a CSV master file.
        for archive in scan_files(INPUT_ROOT,{'.zip'}):
            try:
                with zipfile.ZipFile(archive) as z:
                    for name in z.namelist():
                        if not name.lower().endswith('.csv') or name.startswith('__MACOSX/'):
                            continue
                        with z.open(name) as stream:
                            if csv_header_candidate(stream):
                                source_candidates.append((archive,name))
            except (zipfile.BadZipFile,OSError):
                pass
    if not source_candidates:
        print('INPUT ROOT EXISTS:',INPUT_ROOT.exists())
        for p in scan_files(INPUT_ROOT,{'.csv','.zip','.xlsx','.parquet'})[:70]:
            print('  Input file:',p)
        raise FileNotFoundError(
            'HydroSalNet 349x548 master CSV was not found among Kaggle Inputs. '
            'Attach HydroSalNet via Add Input; if only XLSX exists, upload the final CSV; '
            'then rerun from the first cell. The dataset display name is not a filename.'
        )

# Load only schema-matched sources and verify full shape and label validity.
validated=[]
for p,member in source_candidates:
    try:
        if member is None:
            candidate=pd.read_csv(p,low_memory=False)
        else:
            with zipfile.ZipFile(p) as archive:
                with archive.open(member) as f: candidate=pd.read_csv(f,low_memory=False)
        candidate.columns = candidate.columns.astype(str).str.lstrip('\ufeff')
        valid=(candidate.shape==(EXPECTED_ROWS,EXPECTED_COLUMNS) and
               all(c in candidate.columns for c in REQUIRED_COLUMNS) and
               candidate['sample_id'].is_unique and candidate['sample_id'].notna().all() and
               pd.to_numeric(candidate['EC_1to5_mScm'],errors='coerce').notna().all())
        if valid:
            # Fingerprint actual CSV content, not the enclosing ZIP.
            if member is None: sha=hashlib.sha256(p.read_bytes()).hexdigest()
            else:
                with zipfile.ZipFile(p) as z: sha=hashlib.sha256(z.read(member)).hexdigest()
            validated.append((p,member,sha,candidate))
    except Exception as exc:
        print('Skipped invalid candidate:',p.name,str(exc)[:150])
if not validated:
    raise ValueError('A CSV had the expected headers but failed 349 x 548/ID/EC checks. Inspect the Input files above.')

# If multiple copies are byte-identical, prefer the expected canonical filename.
sha_groups={}
for p,member,sha,c in validated: sha_groups.setdefault(sha,[]).append((p,member,sha,c))
if len(sha_groups)>1:
    info=[f'{p} [{member or "plain CSV"}] SHA256={sha[:12]}' for p,member,sha,c in validated]
    raise ValueError('Multiple DIFFERENT 349x548 masters found. Set DATA_PATH to the correct CSV:\n'+'\n'.join(info))
validated.sort(key=lambda v:(v[0].name != PREFERRED_CSV_NAME,len(str(v[0]))))
input_path,zip_member,DATA_SHA,df=validated[0]
if zip_member is None:
    DATA_PATH=str(input_path)
else:
    # Extract only the selected CSV member to writable workspace for later references.
    dst=WORK_DIR/'source_extracted.csv'
    with zipfile.ZipFile(input_path) as z:
        with z.open(zip_member) as source,open(dst,'wb') as target:
            shutil.copyfileobj(source,target)
    DATA_PATH=str(dst)
print(f'[DATASET FOUND] {input_path}' + (f' :: {zip_member}' if zip_member else ''))
print('[VERIFIED]',df.shape,'SHA256',DATA_SHA[:16]+'...')

# Restore only from user-controlled private checkpoint archive, after selecting the master.
possible_zips=[]
if RESTORE_ZIP_PATH: possible_zips.append(Path(RESTORE_ZIP_PATH))
possible_zips+=scan_files(INPUT_ROOT,{'.zip'})
possible_zips=[p for p in possible_zips if p.name=='hydrosalnet-recovery.zip' or str(p)==RESTORE_ZIP_PATH]
check_dir=WORK_DIR/'checkpoints'
if not any(check_dir.glob('*')):
    for z in possible_zips:
        try:
            with zipfile.ZipFile(z) as archive:
                entries=archive.infolist()
                if 'manifest.json' not in {e.filename for e in entries}:
                    print('[RESTORE] Skipping unrelated ZIP:',z);continue
                mf=json.loads(archive.read('manifest.json'))
                if mf.get('sha256')!=DATA_SHA or mf.get('run')!=RUN_NAME:
                    print('[RESTORE] Skipping ZIP for another dataset or run:',z);continue
                for entry in entries:
                    rel=Path(entry.filename)
                    dest=(WORK_DIR/rel).resolve()
                    if not dest.is_relative_to(WORK_DIR.resolve()) or entry.filename.startswith('/'):
                        raise ValueError('Unsafe ZIP entry')
                archive.extractall(WORK_DIR)
            print('[RESTORED CHECKPOINTS]',z)
            break
        except (zipfile.BadZipFile,OSError,ValueError) as exc:
            print('[RESTORE ERROR]',z,str(exc)[:140])

# Verify target and observation metadata.
df['sample_date']=pd.to_datetime(df['sample_date'],errors='coerce')
assert df['sample_date'].notna().all(), 'Invalid sample dates'
df['EC_1to5_mScm']=pd.to_numeric(df['EC_1to5_mScm'],errors='coerce')
assert np.isfinite(df['EC_1to5_mScm']).all() and (df['EC_1to5_mScm']>=0).all()
assert pd.to_numeric(df['latitude'],errors='coerce').between(20,27).all()
assert pd.to_numeric(df['longitude'],errors='coerce').between(88,93).all()
assert set(df['sample_year'].astype(int).unique())=={2024,2025}
assert (df['sample_date'].dt.year==df['sample_year'].astype(int)).all()
print('Ground sample years:',df.sample_year.value_counts().sort_index().to_dict())
if 'ground_ec_protocol_status' in df:
    print('Ground lab protocol flags:',df.ground_ec_protocol_status.value_counts(dropna=False).to_dict())

# Freeze all architecture and training settings so resume never mixes incompatible configs.
RUN_CONFIG={
  'seed':RANDOM_SEED,'folds':CV_FOLDS,'epochs':EPOCHS,'rounds':BOOSTING_ROUNDS,
  'trees':TREE_COUNT,'chunk':CHECKPOINT_EVERY_TREE_STEP,'hidden':NN_HIDDEN,
  'embed':NN_EMBED,'lr':NN_LR,'weight_decay':NN_WEIGHT_DECAY,
  'batch':NN_BATCH_SIZE,'dropout':NN_DROPOUT,'modality_dropout':MODALITY_DROPOUT,
  'code':'hydrosalnet-kaggle-v2-fixed-20261117'
}
manifest={'run':RUN_NAME,'input_file':Path(DATA_PATH).name,'sha256':DATA_SHA,
          'rows':len(df),'columns':df.shape[1],
          'created_utc':datetime.now(timezone.utc).isoformat(),
          'target':'EC_1to5_mScm','run_configuration':RUN_CONFIG}
manifest_path=WORK_DIR/'manifest.json'
if manifest_path.exists():
    old=json.loads(manifest_path.read_text())
    if old.get('sha256')!=DATA_SHA or old.get('run')!=RUN_NAME or old.get('run_configuration')!=RUN_CONFIG:
        raise RuntimeError('INCOMPATIBLE SAVED RUN: dataset, run name or training settings changed. '
                           'Change RUN_NAME to start a new clean run, or restore matching settings.')
else: atomic_json(manifest_path,manifest)
display(df[REQUIRED_COLUMNS].head())


## 3. Exploratory data analysis — audited data inventory

These charts describe the dataset, but **are not independent proof of model generalization**. Note that 548 source columns contain duplicates, QA, metadata and constants; the modeling selector below excludes these by allowlist.

In [ ]:
# Numeric/constant/missingness statistics and dataset inventory
n_rows,n_columns=df.shape
numeric = df.select_dtypes(include=np.number)
missing_pct = (df.isna().mean()*100).sort_values(ascending=False)
constant_columns=[col for col in df.columns if df[col].nunique(dropna=False)<=1]
all_missing=[col for col in df.columns if df[col].isna().all()]
summary_table=pd.DataFrame({
    "metric":["Total rows","Total columns","2024 samples","2025 samples","Unique IDs",
              "Numeric columns","Columns with any missing values","Fully empty columns",
              "Constant columns incl. missing","EC missing"],
    "value":[n_rows,n_columns,int((df.sample_year==2024).sum()),int((df.sample_year==2025).sum()),
             df.sample_id.nunique(),numeric.shape[1],int((missing_pct>0).sum()),
             len(all_missing),len(constant_columns),int(df.EC_1to5_mScm.isna().sum())]
})
display(summary_table)
save_csv_atomic(summary_table,WORK_DIR/"eda"/"dataset_inventory.csv")
save_csv_atomic(pd.DataFrame({"column":missing_pct.index,"missing_pct":missing_pct.values}),WORK_DIR/"eda"/"missingness_by_column.csv")
save_csv_atomic(pd.DataFrame({"constant_column":constant_columns}),WORK_DIR/"eda"/"constant_columns.csv")

display(df.groupby("sample_year")["EC_1to5_mScm"].agg(["count","mean","median","std","min","max"]).round(3))
print("CAUTION: 2024 and 2025 are different campaigns/months/locations. Their means cannot establish yearly salinity change.")

In [ ]:
# Figure 1: Campaign sample count (bar), district distribution (stacked bar), and land-use class (pie)
fig,ax=plt.subplots(1,3,figsize=(16,4.6))
df["sample_year"].value_counts().sort_index().plot.bar(ax=ax[0],rot=0,title="Ground EC samples by campaign")
ax[0].set_ylabel("Number of samples")
if "ground_district" in df:
    pd.crosstab(df["ground_district"].fillna("Unknown"),df["sample_year"]).plot.bar(ax=ax[1],stacked=True)
    ax[1].set_title("District coverage × campaign");ax[1].set_ylabel("Samples");ax[1].tick_params(axis='x',rotation=35)
else: ax[1].text(.1,.5,"No district field")
if "DW_class" in df:
    top=df["DW_class"].fillna("Unknown").value_counts(); show=top.head(6)
    if len(top)>6: show.loc["Other classes"]=top.iloc[6:].sum()
    ax[2].pie(show.values,labels=show.index,autopct="%1.0f%%",textprops={"fontsize":8})
    ax[2].set_title("Dynamic World mapped class")
else: ax[2].text(.1,.5,"No Dynamic World class")
fig.suptitle("HydroSalNet: study inventory and mapped land-cover context",fontsize=13)
savefig("01_campaign_district_landcover")

In [ ]:
# Figure 2: EC distributions and campaign comparison (different sampling months!)
fig,ax=plt.subplots(2,2,figsize=(14,9))
sns.histplot(data=df,x="EC_1to5_mScm",hue="sample_year",kde=True,ax=ax[0,0],bins=24,element="step")
ax[0,0].set_title("EC1:5 distribution by campaign")
sns.boxplot(data=df,x="sample_year",y="EC_1to5_mScm",ax=ax[0,1])
ax[0,1].set_title("EC1:5 by campaign: different months / locations")
sns.violinplot(data=df,x="sample_year",y="EC_1to5_mScm",inner="quartile",ax=ax[1,0])
ax[1,0].set_title("Campaign EC distribution (violin)")
sns.ecdfplot(data=df,x="EC_1to5_mScm",hue="sample_year",ax=ax[1,1])
ax[1,1].set_title("Empirical cumulative EC distribution")
fig.suptitle("Measured laboratory soil EC1:5 (mS/cm)",fontsize=13)
savefig("02_ec_distributions")

In [ ]:
# Figure 3: Geographic sampling scatter and measured EC at point locations (NOT raster salinity map)
fig,ax=plt.subplots(1,2,figsize=(14,5))
for y,g in df.groupby('sample_year'):
    ax[0].scatter(g.longitude,g.latitude,label=str(y),s=33,alpha=.8)
ax[0].legend();ax[0].set_title("Sampling coordinates by year")
pts=ax[1].scatter(df.longitude,df.latitude,c=df.EC_1to5_mScm,cmap="viridis",s=38,alpha=.9)
plt.colorbar(pts,ax=ax[1],label="Measured EC1:5 (mS/cm)")
ax[1].set_title("Measured EC at sample points (not a prediction map)")
for a in ax:
    a.set_xlabel("Longitude (WGS84)");a.set_ylabel("Latitude (WGS84)")
savefig("03_geo_points_ec")

In [ ]:
# Figure 4: Missingness, source presence, optical QA (real S30 gaps)
quality_presence={
    "S1 SAR": "S1_VV_dB", "S2 optical":"S2_NDVI", "HLS L30":"L30_NDVI",
    "HLS S30":"S30_NDVI", "CHIRPS":"CHIRPS_rain_30d_mm", "ERA5":"ERA5_temp_mean_30d_C",
    "SoilGrids direct":"SG_clay_pct_0_15", "SoilGrids modeled":"SG_model_clay_pct_0_15",
    "SMAP":"SMAP_surface_mean_30d", "GloFAS":"GLOFAS_mean_30d_cms",
    "TerraClimate":"TC_pet_sum_3m", "GPM":"GPM_total_mm_30d", "WorldCereal":"WC2021_confidence",
    "HydroRIVERS":"HYR_distance_nearest_m", "IBTrACS":"IB_nearest_center_km_3y"}
coverage={k:100*df[v].notna().mean() for k,v in quality_presence.items() if v in df}
fig,ax=plt.subplots(1,2,figsize=(15,6))
pd.Series(coverage).sort_values().plot.barh(ax=ax[0],xlim=(0,105),title="Observed feature coverage by data source")
ax[0].set_xlabel("Available records (%)")
missing_pct.head(25).iloc[::-1].plot.barh(ax=ax[1],title="25 highest-missing source columns")
ax[1].set_xlabel("Missing records (%)")
savefig("04_source_coverage_and_missingness")

fig,ax=plt.subplots(1,2,figsize=(13,4.8))
if "S30_status" in df:
    df.S30_status.value_counts().plot.pie(ax=ax[0],autopct="%1.1f%%",ylabel="",title="HLS S30 availability")
else: ax[0].set_axis_off()
if "L30_water_quality" in df:
    df.L30_water_quality.fillna("Missing").value_counts().plot.pie(ax=ax[1],autopct="%1.1f%%",ylabel="",title="HLS L30 water-quality class")
else: ax[1].set_axis_off()
savefig("05_optical_quality_pies")

In [ ]:
# Figure 5: Rainfall by window — line graph; campaign-grouped, NOT a multi-year measured trend
rain_cols=["CHIRPS_rain_7d_mm","CHIRPS_rain_30d_mm","CHIRPS_rain_90d_mm"]
gpm_cols=["GPM_total_mm_7d","GPM_total_mm_30d","GPM_total_mm_90d"]
fig,ax=plt.subplots(1,2,figsize=(14,4.8))
for year,g in df.groupby('sample_year'):
    ax[0].plot([7,30,90],g[rain_cols].mean().values,marker='o',label=f"CHIRPS {year}")
    ax[0].plot([7,30,90],g[gpm_cols].mean().values,marker='s',linestyle='--',label=f"GPM {year}")
ax[0].legend();ax[0].set_xlabel("Pre-sampling days");ax[0].set_ylabel("Mean accumulated rain (mm)")
ax[0].set_title("Antecedent rainfall (different products)")
short=["S2_NDVI","S1_VV_dB","ERA5_temp_mean_30d_C","SMAP_surface_mean_30d"]
for c in short:
    if c in df: ax[1].scatter(df[c],df["EC_1to5_mScm"],s=16,alpha=.45,label=c)
ax[1].legend(fontsize=7);ax[1].set_title("Different scales: exploratory scatter only")
ax[1].set_xlabel("Raw covariate value (heterogeneous units)");ax[1].set_ylabel("EC1:5")
savefig("06_antecedent_rainfall_lines")

## 4. Leakage-screened modeling features

**Only columns explicitly named below may enter the models.** This guards against accidentally including EC, sample IDs, coordinates, year, dates, product labels, source-quality summary strings or exact numeric target duplicates. Some values are highly correlated—later experiments examine this via ablation; 548 columns are *not* 548 independent predictors.

For quality-aware fusion, the `S2`, `L30` and `S30` branches remain separate so real missingness can be learned. Weather, soil, hydrology and land-cover are coarser or partly historical proxies, not direct soil salinity measurements.

In [ ]:
GROUPS=OrderedDict({
    "S2_optical": ["S2_B2","S2_B4","S2_B8","S2_B11","S2_NDVI","S2_NDMI","S2_BSI"],
    "L30_optical": ["L30_red","L30_nir","L30_swir1","L30_NDVI","L30_NDMI","L30_BSI"],
    "S30_optical": ["S30_red","S30_nir","S30_swir1","S30_NDVI","S30_NDMI","S30_NDRE"],
    "SAR": ["S1_VV_dB","S1_VH_dB"],
    "Climate": ["CHIRPS_rain_7d_mm","CHIRPS_rain_30d_mm","CHIRPS_rain_90d_mm",
                "GPM_total_mm_7d","GPM_total_mm_30d", "ERA5_temp_mean_30d_C",
                "ERA5_soil_moisture_0_7cm_30d", "ERA5_evap_total_30d_mm",
                "TC_def_sum_3m","TC_pet_sum_3m","TC_soil_storage_mean_3m"],
    "Soil": ["SG_model_clay_pct_0_15","SG_model_sand_pct_0_15","SG_model_pH_0_15",
             "SG_model_SOC_gkg_0_15","SG_model_bulk_density_gcm3_0_15",
             "SMAP_surface_mean_30d","SMAP_rootzone_mean_30d"],
    "Hydrology": ["GLOFAS_mean_30d_cms","GLOFAS_max_90d_cms","HYR_distance_nearest_m",
                  "HYR_nearest_longterm_discharge_cms","HYR_mainstem_distance_m",
                  "OSM_dist_waterway_m","IB_storm_count_250km_10y"],
    "Terrain_landcover": ["DEM_elevation_m","DEM_slope_deg","DW_water","DW_crops",
                          "DW_bare","WC2021_crop_fraction_250m","OSM_dist_road_m"]
})

missing_selected=[x for group in GROUPS.values() for x in group if x not in df]
if missing_selected:
    raise ValueError("Exact selected columns missing; audit before continuing: "+repr(missing_selected))
FEATURES=[c for grp in GROUPS.values() for c in grp]
assert len(FEATURES)==len(set(FEATURES)),"Duplicate feature in groups"
assert not any(re.search(r'ec_|salinity|label|sample_id|longitude|latitude|year|date',c,re.I) for c in FEATURES),"Leakage candidate"
for c in FEATURES: df[c]=pd.to_numeric(df[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
print("Environmental predictor count:",len(FEATURES))
print("Groups:", {k:len(v) for k,v in GROUPS.items()})
print("Overall missing predictor cells:",round(df[FEATURES].isna().mean().mean()*100,2),"%")
print("Fully missing training columns will be filled with zero after training-fold imputer; never use held-out medians.")
atomic_json(WORK_DIR/"feature_manifest.json", {"groups":GROUPS,"features":FEATURES,"dataset_sha256":DATA_SHA})
# Prevent mixing saved checkpoints with a different feature/schema definition.
feature_signature=hashlib.sha256(json.dumps(GROUPS,sort_keys=True).encode()).hexdigest()
feature_state=WORK_DIR/'feature_signature.json'
if feature_state.exists():
    prev=json.loads(feature_state.read_text())
    if prev.get('sha')!=feature_signature:
        raise RuntimeError('FEATURE GROUPS CHANGED: choose a NEW RUN_NAME instead of resuming old checkpoints.')
else: atomic_json(feature_state,{'sha':feature_signature,'features':FEATURES})


In [ ]:
# EDA: feature group distributions, EC correlation, and numeric correlation matrix
num_selected=df[FEATURES].copy()
fig,ax=plt.subplots(1,2,figsize=(16,6))
missing_by_group={k:100*df[v].isna().mean().mean() for k,v in GROUPS.items()}
pd.Series(missing_by_group).sort_values().plot.barh(ax=ax[0],title="Missing cells by modeling feature group")
ax[0].set_xlabel("Missing cells (%)")
correlations=num_selected.corrwith(df.EC_1to5_mScm).dropna().sort_values()
if len(correlations)>0:
    pd.concat([correlations.head(10),correlations.tail(10)]).sort_values().plot.barh(ax=ax[1],title="Exploratory Pearson correlation with EC")
ax[1].set_xlabel("Correlation (not causal importance)")
savefig("07_feature_groups_correlations")

corr_cols=[c for c in ["S2_NDVI","L30_NDVI","S30_NDVI","S1_VV_dB","CHIRPS_rain_30d_mm",
                       "GPM_total_mm_30d","ERA5_temp_mean_30d_C","SMAP_surface_mean_30d",
                       "SG_model_clay_pct_0_15","GLOFAS_mean_30d_cms","DEM_elevation_m",
                       "HYR_distance_nearest_m","EC_1to5_mScm"] if c in df]
fig,ax=plt.subplots(figsize=(12,9))
sns.heatmap(df[corr_cols].corr(),cmap="coolwarm",center=0,vmin=-1,vmax=1,annot=True,fmt=".2f",ax=ax,annot_kws={"size":7})
ax.set_title("Selected numerical correlation heatmap (exploratory)")
savefig("08_selected_correlation_heatmap")

if "WC2021_temp_crop_binary" in df:
    fig,ax=plt.subplots(figsize=(8,4.5))
    sns.boxplot(data=df,x="WC2021_temp_crop_binary",y="EC_1to5_mScm",ax=ax)
    ax.set_title("EC by historical 2021 temporary-crop class (NOT a 2024/25 measurement)")
    savefig("09_worldcereal_ec_context")

## 5. Construct predefined validation protocols — before training

- **Random 4-fold**: diagnostic only; may exaggerate deployment accuracy.
- **Spatial 4-fold**: use contiguous-like geographic KMeans clusters in *2024 training data only*. Other block definitions and buffered spatial holdouts must be tested before publication.
- **2024→2025**: 2025 is a sealed **final campaign holdout**; do not use the scores to change training choices. Cross-campaign mixes month, geography and year.

**All imputation/scaling is fitted on training rows for each fold**, never on CV test or 2025 labels.

In [ ]:
idx24=np.where(df.sample_year.astype(int).to_numpy()==2024)[0]
idx25=np.where(df.sample_year.astype(int).to_numpy()==2025)[0]
Xraw=df[FEATURES].to_numpy(dtype=np.float32)
y=df.EC_1to5_mScm.to_numpy(dtype=np.float32)

# Convert degrees to a local approximate metric coordinate system for block construction only.
lat0=float(df.loc[idx24,'latitude'].mean())
coords=np.c_[df.latitude.to_numpy()*111.3,df.longitude.to_numpy()*111.3*np.cos(np.deg2rad(lat0))]
blocks=KMeans(n_clusters=CV_FOLDS,random_state=RANDOM_SEED,n_init=10).fit_predict(coords[idx24])
print("2024 spatial cluster sizes:",dict(Counter(blocks)))
if min(Counter(blocks).values())<5:
    warnings.warn("At least one spatial block has <5 samples; test other block sizes for publication")
random_kf=KFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_SEED)
random_splits=[(idx24[tr],idx24[te],f"random_{i}") for i,(tr,te) in enumerate(random_kf.split(idx24),1)]
spatial_gkf=GroupKFold(n_splits=CV_FOLDS)
spatial_splits=[(idx24[tr],idx24[te],f"spatial_{i}") for i,(tr,te) in enumerate(spatial_gkf.split(idx24,y[idx24],groups=blocks),1)]
forward_splits=[(idx24,idx25,"forward_2024_to_2025")]
for label,splits in [("random",random_splits),("spatial",spatial_splits)]:
    for tr,te,fold in splits:
        assert not set(tr)&set(te)
        assert (df.iloc[tr].sample_year==2024).all() and (df.iloc[te].sample_year==2024).all()
assert not set(idx24)&set(idx25)

# Save immutable fold membership BEFORE any ML preprocessing.
fold_rows=[]
for proto,splits in [("random",random_splits),("spatial",spatial_splits),("forward",forward_splits)]:
    for tr,te,fold in splits:
        fold_rows += [{"protocol":proto,"fold":fold,"sample_id":df.iloc[i].sample_id,"role":"train"} for i in tr]
        fold_rows += [{"protocol":proto,"fold":fold,"sample_id":df.iloc[i].sample_id,"role":"test"} for i in te]
split_path=WORK_DIR/'splits_frozen.csv'
new_folds=pd.DataFrame(fold_rows)
if split_path.exists():
    previous_folds=pd.read_csv(split_path,dtype={'sample_id':str})
    if not previous_folds.equals(new_folds.reset_index(drop=True)):
        raise RuntimeError('SPATIAL/RANDOM FOLDS CHANGED: choose a new RUN_NAME to avoid mixing results.')
else: save_csv_atomic(new_folds,split_path)

fig,ax=plt.subplots(figsize=(8,5.5))
sc=ax.scatter(df.iloc[idx24].longitude,df.iloc[idx24].latitude,c=blocks,cmap="tab10",s=34)
plt.colorbar(sc,ax=ax,label="2024 geographic block")
ax.set_title("Spatial validation groups (2024 training campaign only)")
ax.set_xlabel("Longitude");ax.set_ylabel("Latitude")
savefig("10_spatial_folds")
print("Validation folds saved:",WORK_DIR/"splits_frozen.csv")

## 6. Training-only preprocessing and modality-quality signals

For the gated network, quality is defined using **observed sensor metadata rather than EC labels**. The gate can learn to ignore poor observations, but its weights are not a causal interpretation of salinity mechanisms.

Quality-aware and quality-blind MLPs receive the same environmental predictors; the quality-aware MLP additionally receives exactly the same group-quality and availability information as the proposed gate.

In [ ]:
GROUP_NAMES=list(GROUPS.keys())
GROUP_INDEX={g:np.array([FEATURES.index(c) for c in cols],dtype=int) for g,cols in GROUPS.items()}

def numeric_series(frame, name, default=0.):
    if name not in frame: return np.full(len(frame),default,dtype=np.float32)
    return pd.to_numeric(frame[name],errors='coerce').fillna(default).to_numpy(dtype=np.float32)

def quality_arrays(frame):
    # Quality features are sampled and bounded [0,1] for stable gated training.
    def cap(a): return np.clip(np.nan_to_num(a,nan=0,posinf=0,neginf=0),0,1).astype(np.float32)
    q={
       "S2_optical":cap(numeric_series(frame,'S2_valid_obs_count')/5),
       "L30_optical":cap(numeric_series(frame,'L30_valid_obs')/5)*cap(1-numeric_series(frame,'L30_water_fraction',1)),
       "S30_optical":cap(numeric_series(frame,'S30_valid_obs')/5)*cap(1-numeric_series(frame,'S30_water_fraction',1)),
       "SAR":cap(numeric_series(frame,'S1_image_count')/10),
       "Climate":np.ones(len(frame),dtype=np.float32),
       "Soil":cap(numeric_series(frame,'SG_model_all_six_present'))*(1-0.25*cap(numeric_series(frame,'SG_proxy_found'))),
       "Hydrology":cap(numeric_series(frame,'GLOFAS_all_windows_available',1))*cap(1-numeric_series(frame,'GLOFAS_grid_offset_m')/20000),
       "Terrain_landcover":np.ones(len(frame),dtype=np.float32)
    }
    available=np.column_stack([frame[GROUPS[g]].notna().any(axis=1).to_numpy().astype(np.float32) for g in GROUP_NAMES])
    quality=np.column_stack([q[g] for g in GROUP_NAMES]).astype(np.float32)
    return quality,available

Q,AV=quality_arrays(df)
print("Quality matrix:",Q.shape,"Available matrix:",AV.shape)
print(pd.DataFrame({"branch":GROUP_NAMES,"available_pct":100*AV.mean(axis=0),"mean_quality":Q.mean(axis=0)}).to_string(index=False))

class TrainOnlyPreprocessor:
    def __init__(self):
        self.imputer=SimpleImputer(strategy='median',keep_empty_features=True)
        self.scaler=StandardScaler()
    def fit(self,raw):
        a=self.imputer.fit_transform(raw)
        self.scaler.fit(a)
        return self
    def transform(self,raw):
        val=self.scaler.transform(self.imputer.transform(raw))
        return np.nan_to_num(val,nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)

def get_inputs(tr,te):
    p=TrainOnlyPreprocessor().fit(Xraw[tr])
    xtr=p.transform(Xraw[tr]);xte=p.transform(Xraw[te])
    return p,xtr,xte

# The optional quality vector + branch-present mask, never IDs or dates.
def augment(x,indices): return np.concatenate([x,Q[indices],AV[indices]],axis=1).astype(np.float32)

## 7. Crash recovery: atomic local checkpoints + optional private Kaggle Dataset snapshots

Neural model files contain **model weights + optimizer state + epoch number + RNG state**; tree-based files include their current estimator state. A failed session can resume from the latest **surviving** checkpoint. Local snapshots are written automatically, but cross-session recovery needs a **remote snapshot** or a previously completed Kaggle notebook version.

To enable remote recovery, add the Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` with notebook access, enable Internet, and set `ENABLE_REMOTE_BACKUP=True`. No raw CSV is uploaded by the backup code. **Only trusted, privately stored checkpoint packages should be reloaded**.

In [ ]:
# Remotely publish only weights/results, never the source ground dataset.
_remote_last=0.
_remote_disabled=False
_remote_error_count=0

def configure_kaggle_auth():
    if not ENABLE_REMOTE_BACKUP: return False
    try:
        from kaggle_secrets import UserSecretsClient
        client=UserSecretsClient()
        user=client.get_secret('KAGGLE_USERNAME')
        key=client.get_secret('KAGGLE_KEY')
        assert user and key, "Missing Kaggle username or API token"
        os.environ['KAGGLE_USERNAME']=user
        os.environ['KAGGLE_KEY']=key
        if importlib.util.find_spec('kaggle') is None:
            subprocess.check_call([sys.executable,'-m','pip','install','-q','kaggle'],timeout=120)
        return True
    except Exception as e:
        print('Remote backup NOT configured:',type(e).__name__,str(e)[:130])
        print('Training will still save local checkpoints; unexpected session failure may lose them.')
        return False

_REMOTE_AUTH_OK=configure_kaggle_auth()
print('Remote backup status:', 'ENABLED (private uploads)' if _REMOTE_AUTH_OK else 'DISABLED (local only)')

RECOVERY_FILES=['manifest.json','feature_manifest.json','feature_signature.json','splits_frozen.csv']
def make_recovery_zip():
    output=WORK_DIR/'hydrosalnet-recovery.zip'
    tmp=WORK_DIR/'hydrosalnet-recovery.zip.tmp'
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
        for folder in ['checkpoints','models','results','logs']:
            for path in (WORK_DIR/folder).rglob('*'):
                if path.is_file() and not path.name.endswith('.tmp'):
                    z.write(path,path.relative_to(WORK_DIR).as_posix())
        for file in RECOVERY_FILES:
            p=WORK_DIR/file
            if p.is_file():z.write(p,file)
    os.replace(tmp,output)
    return output

def remote_sync(force=False,why='checkpoint'):
    global _remote_last,_remote_error_count,_remote_disabled
    if not _REMOTE_AUTH_OK or _remote_disabled:return False
    if not force and time.monotonic()-_remote_last < REMOTE_MIN_SECONDS_BETWEEN_UPLOADS:return False
    try:
        recovery=make_recovery_zip()
        stage=WORK_DIR/'_remote_stage'; stage.mkdir(exist_ok=True)
        target=stage/recovery.name
        shutil.copy2(recovery,target)
        owner=os.environ['KAGGLE_USERNAME']
        metadata={'title':'HydroSalNet Private Checkpoints','id':owner+'/'+REMOTE_DATASET_SLUG,
                  'licenses':[{'name':'other'}],
                  'description':'Private model checkpoints and training status only; no raw ground samples.'}
        (stage/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
        cli=[sys.executable,'-m','kaggle']
        if (WORK_DIR/'remote_dataset_created.flag').exists():
            cmd=cli+['datasets','version','-p',str(stage),'-m',why,'-q']
        else:
            cmd=cli+['datasets','create','-p',str(stage),'-q']
        run=subprocess.run(cmd,text=True,capture_output=True,timeout=240)
        if run.returncode !=0 and 'create' in cmd:
            # A prior crashed session may already have created the private dataset.
            cmd=cli+['datasets','version','-p',str(stage),'-m',why,'-q']
            run=subprocess.run(cmd,text=True,capture_output=True,timeout=240)
        if run.returncode!=0: raise RuntimeError((run.stderr or run.stdout)[-300:])
        (WORK_DIR/'remote_dataset_created.flag').write_text(metadata['id'])
        _remote_last=time.monotonic();_remote_error_count=0
        print('[PRIVATE REMOTE BACKUP OK]',why,'->',metadata['id'])
        return True
    except Exception as e:
        _remote_error_count+=1
        print('[REMOTE BACKUP FAILED] Retaining local files:',type(e).__name__,str(e)[:200])
        if _remote_error_count>=6:
            print('Too many backup failures; disabling remote uploads for this session.')
            _remote_disabled=True
        return False

# Save initial source and run info to remote as a readiness test.
if ENABLE_REMOTE_BACKUP:
    if not _REMOTE_AUTH_OK or not remote_sync(force=True,why='Initial run metadata before training'):
        raise RuntimeError('REMOTE BACKUP REQUESTED but verification failed. Check Internet, Kaggle Secrets, '
                           'API permissions and Kaggle private dataset quota. Disable backup explicitly '
                           'only if you accept local checkpoints may be lost in a Kaggle session failure.')
    print('[REMOTE BACKUP PREFLIGHT VERIFIED] Initial private dataset upload succeeded.')

## 8. Model training: 50 trees/boosting rounds or 50 neural epochs

Train **all candidate models with a fixed budget** and checkpoints. Checkpointed CV folds will be skipped when rerunning with the same dataset hash and run name. Model selection is based on **2024 spatial validation**, not 2025 final metrics.

In [ ]:
MODEL_NAMES=['DummyMean','Ridge','ElasticNet','RandomForest','ExtraTrees','XGBoost','CatBoost',
             'LightGBM','MLP_NoQA','MLP_EarlyFusion_QA','HydroSalNet_Gated']
METRIC_FILE=WORK_DIR/'results'/'all_fold_metrics.csv'
PRED_FILE=WORK_DIR/'results'/'out_of_fold_predictions.csv'
LEARNING_FILE=WORK_DIR/'logs'/'learning_history.csv'
if not METRIC_FILE.exists():pd.DataFrame(columns=['protocol','fold','model','n_train','n_test','mae','rmse','r2','bias','data_sha']).to_csv(METRIC_FILE,index=False)
if not PRED_FILE.exists():pd.DataFrame(columns=['protocol','fold','model','sample_id','year','truth','predicted']).to_csv(PRED_FILE,index=False)
if not LEARNING_FILE.exists():pd.DataFrame(columns=['protocol','fold','model','step','train_mae','train_loss']).to_csv(LEARNING_FILE,index=False)

def append_row(filename,dct):
    pd.DataFrame([dct]).to_csv(filename,mode='a',index=False,header=False)

def recorded(protocol,fold,model):
    frame=pd.read_csv(METRIC_FILE)
    if len(frame)==0:return False
    return bool(((frame.protocol==protocol)&(frame.fold==fold)&(frame.model==model)&(frame.data_sha==DATA_SHA)).any())

def accuracy(y_true,yp):
    return {'mae':float(mean_absolute_error(y_true,yp)),
            'rmse':float(np.sqrt(mean_squared_error(y_true,yp))),
            'r2':float(r2_score(y_true,yp)),
            'bias':float(np.mean(np.asarray(yp)-np.asarray(y_true)))}

def checkpoint_file(protocol,fold,model,suffix):
    return WORK_DIR/'checkpoints'/f'{protocol}__{fold}__{model}.{suffix}'

# Input data to models contains ONLY train-fold-fitted transforms.
def fit_tree_or_linear(model_name,protocol,fold,tr,te,p,xtr,xte):
    yt=y[tr]
    pth=checkpoint_file(protocol,fold,model_name,'joblib')
    num_stages= TREE_COUNT if model_name in ['RandomForest','ExtraTrees'] else BOOSTING_ROUNDS
    if model_name in ['DummyMean','Ridge','ElasticNet']:
        if pth.exists():
            obj=joblib.load(pth); assert obj.get('sha')==DATA_SHA; mdl=obj['model']
        else:
            if model_name=='DummyMean':mdl=DummyRegressor(strategy='mean')
            elif model_name=='Ridge':mdl=Ridge(alpha=10.)
            else: mdl=ElasticNet(alpha=.12,l1_ratio=.4,max_iter=5000,random_state=RANDOM_SEED)
            mdl.fit(xtr,yt);save_joblib_atomic({'model':mdl,'step':1,'sha':DATA_SHA},pth)
        return mdl.predict(xte)

    if model_name in ['RandomForest','ExtraTrees']:
        if pth.exists(): obj=joblib.load(pth); assert obj['sha']==DATA_SHA; mdl=obj['model']; done=obj['step']
        else:
            klass=RandomForestRegressor if model_name=='RandomForest' else ExtraTreesRegressor
            mdl=klass(n_estimators=1,warm_start=True,min_samples_leaf=3,max_features=.8,
                      random_state=RANDOM_SEED,n_jobs=2);done=0
        while done<num_stages:
            n=min(num_stages,done+CHECKPOINT_EVERY_TREE_STEP)
            mdl.set_params(n_estimators=n);mdl.fit(xtr,yt)
            done=n
            save_joblib_atomic({'model':mdl,'step':done,'sha':DATA_SHA},pth)
            append_row(LEARNING_FILE,{'protocol':protocol,'fold':fold,'model':model_name,'step':done,
                    'train_mae':float(mean_absolute_error(yt,mdl.predict(xtr))),'train_loss':np.nan})
        return mdl.predict(xte)

    if model_name=='XGBoost':
        path=checkpoint_file(protocol,fold,model_name,'json')
        state=checkpoint_file(protocol,fold,model_name,'state.json')
        done=json.loads(state.read_text())['step'] if path.exists() and state.exists() else 0
        if path.exists() and state.exists():
            assert json.loads(state.read_text()).get('sha')==DATA_SHA,'XGBoost checkpoint mismatch'
        dtrain=xgb.DMatrix(xtr,label=yt,feature_names=FEATURES)
        dtest=xgb.DMatrix(xte,feature_names=FEATURES)
        params={'objective':'reg:squarederror','max_depth':3,'eta':.06,'subsample':.85,
                'colsample_bytree':.9,'lambda':5.0,'seed':RANDOM_SEED,'nthread':2}
        booster=xgb.Booster(model_file=str(path)) if path.exists() else None
        # Model file is written before step JSON; actual model rounds are authoritative.
        if booster is not None:
            done=int(booster.num_boosted_rounds())
        while done<BOOSTING_ROUNDS:
            step=min(CHECKPOINT_EVERY_TREE_STEP,BOOSTING_ROUNDS-done)
            booster=xgb.train(params,dtrain,num_boost_round=step,xgb_model=booster,verbose_eval=False)
            tmp=Path(str(path)+'.tmp.json');booster.save_model(str(tmp))
            os.replace(tmp,path);done+=step
            atomic_json(state,{'step':done,'sha':DATA_SHA})
            append_row(LEARNING_FILE,{'protocol':protocol,'fold':fold,'model':model_name,'step':done,
                'train_mae':float(mean_absolute_error(yt,booster.predict(dtrain))),'train_loss':np.nan})
        return booster.predict(dtest)

    if model_name=='CatBoost':
        path=checkpoint_file(protocol,fold,model_name,'cbm')
        state=checkpoint_file(protocol,fold,model_name,'state.json')
        done=json.loads(state.read_text())['step'] if path.exists() and state.exists() else 0
        if path.exists() and state.exists():
            assert json.loads(state.read_text()).get('sha')==DATA_SHA,'CatBoost checkpoint mismatch'
        previous=None
        if path.exists():
            previous=cb.CatBoostRegressor();previous.load_model(str(path))
            done=int(previous.tree_count_)
        while done<BOOSTING_ROUNDS:
            step=min(CHECKPOINT_EVERY_TREE_STEP,BOOSTING_ROUNDS-done)
            mdl=cb.CatBoostRegressor(iterations=step,learning_rate=.06,depth=4,l2_leaf_reg=6,
                   loss_function='RMSE',random_seed=RANDOM_SEED,verbose=False,
                   allow_writing_files=False,thread_count=2)
            mdl.fit(xtr,yt,init_model=previous)
            tmp=Path(str(path)+'.tmp');mdl.save_model(str(tmp),format='cbm')
            os.replace(tmp,path);previous=mdl;done+=step
            atomic_json(state,{'step':done,'sha':DATA_SHA})
            append_row(LEARNING_FILE,{'protocol':protocol,'fold':fold,'model':model_name,'step':done,
                'train_mae':float(mean_absolute_error(yt,mdl.predict(xtr))),'train_loss':np.nan})
        return previous.predict(xte).reshape(-1)

    if model_name=='LightGBM':
        path=checkpoint_file(protocol,fold,model_name,'txt')
        state=checkpoint_file(protocol,fold,model_name,'state.json')
        done=json.loads(state.read_text())['step'] if path.exists() and state.exists() else 0
        if path.exists() and state.exists():
            assert json.loads(state.read_text()).get('sha')==DATA_SHA,'LightGBM checkpoint mismatch'
        previous=lgb.Booster(model_file=str(path)) if path.exists() else None
        if previous is not None: done=int(previous.current_iteration())
        train_set=lgb.Dataset(xtr,label=yt,free_raw_data=False)
        params={'objective':'regression','metric':'l1','learning_rate':.06,'num_leaves':7,
                'min_data_in_leaf':6,'feature_fraction':.9,'lambda_l2':5.0,
                'verbosity':-1,'num_threads':2,'seed':RANDOM_SEED}
        while done<BOOSTING_ROUNDS:
            step=min(CHECKPOINT_EVERY_TREE_STEP,BOOSTING_ROUNDS-done)
            booster=lgb.train(params,train_set,num_boost_round=step,
                              init_model=previous,keep_training_booster=True)
            tmp=Path(str(path)+'.tmp.json');booster.save_model(str(tmp))
            os.replace(tmp,path);previous=booster;done+=step
            atomic_json(state,{'step':done,'sha':DATA_SHA})
            append_row(LEARNING_FILE,{'protocol':protocol,'fold':fold,'model':model_name,'step':done,
                'train_mae':float(mean_absolute_error(yt,booster.predict(xtr))),'train_loss':np.nan})
        return previous.predict(xte)
    raise ValueError(model_name)

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self,in_features):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_features,NN_HIDDEN),nn.ReLU(),nn.Dropout(NN_DROPOUT),
                               nn.Linear(NN_HIDDEN,NN_HIDDEN//2),nn.ReLU(),nn.Linear(NN_HIDDEN//2,1))
    def forward(self,x,q=None,av=None):return self.net(x).squeeze(-1)

class QualityAwareGatedNet(nn.Module):
    """Lightweight branch-wise gated fusion; missing modalities are masked before softmax."""
    def __init__(self):
        super().__init__()
        self.group_names=list(GROUP_NAMES)
        self.encoders=nn.ModuleDict()
        self.gates=nn.ModuleDict()
        for name in self.group_names:
            n=len(GROUP_INDEX[name])
            self.encoders[name]=nn.Sequential(nn.Linear(n,NN_HIDDEN//2),nn.ReLU(),nn.Dropout(NN_DROPOUT),
                                             nn.Linear(NN_HIDDEN//2,NN_EMBED),nn.ReLU())
            self.gates[name]=nn.Sequential(nn.Linear(NN_EMBED+2,12),nn.Tanh(),nn.Linear(12,1))
        self.regressor=nn.Sequential(nn.Linear(NN_EMBED,16),nn.ReLU(),nn.Linear(16,1))
        self.last_gate_weights=None
    def forward(self,x,q,av):
        embeds=[];logits=[];present=av.clone()
        if self.training and MODALITY_DROPOUT>0:
            # The 3 optical branches alone are randomly removed; static terrain always remains.
            for j,name in enumerate(self.group_names):
                if name in ('S2_optical','L30_optical','S30_optical'):
                    dropout=(torch.rand(x.shape[0],device=x.device)<MODALITY_DROPOUT)
                    present[:,j]=present[:,j]*(~dropout).float()
        for j,name in enumerate(self.group_names):
            emb=self.encoders[name](x[:,GROUP_INDEX[name].tolist()])
            logit=self.gates[name](torch.cat([emb,q[:,j:j+1],present[:,j:j+1]],dim=1))
            embeds.append(emb);logits.append(logit)
        stacked=torch.stack(embeds,dim=1)
        raw_logits=torch.cat(logits,dim=1).masked_fill(present<.5,-1e4)
        weights=torch.softmax(raw_logits,dim=1)
        self.last_gate_weights=weights.detach()
        return self.regressor((stacked*weights.unsqueeze(-1)).sum(1)).squeeze(-1)


def neural_inputs(model_name,x,indices):
    return augment(x,indices) if model_name=='MLP_EarlyFusion_QA' else x


def fit_neural(model_name,protocol,fold,tr,te,p,xtr,xte):
    if model_name=='MLP_EarlyFusion_QA':
        features_train=augment(xtr,tr); features_test=augment(xte,te)
    else: features_train=xtr;features_test=xte
    model=QualityAwareGatedNet() if model_name=='HydroSalNet_Gated' else SmallMLP(features_train.shape[1])
    model=model.to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=NN_LR,weight_decay=NN_WEIGHT_DECAY)
    ckpt=checkpoint_file(protocol,fold,model_name,'pt')
    losses=[]; start_epoch=0
    if ckpt.exists():
        saved=torch.load(ckpt,map_location='cpu',weights_only=False)  # trusted local/own private checkpoint
        if saved.get('sha')!=DATA_SHA or saved.get('run')!=RUN_NAME:
            raise RuntimeError('Neural checkpoint fingerprint mismatch')
        model.load_state_dict(saved['model_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        for state in optimizer.state.values():
            for k,v in state.items():
                if torch.is_tensor(v):state[k]=v.to(DEVICE)
        start_epoch=int(saved['epoch']); losses=saved['losses']
        # Restore random stream so resumed dropout closely follows an uninterrupted run.
        if 'torch_rng_state' in saved: torch.set_rng_state(saved['torch_rng_state'])
        if torch.cuda.is_available() and saved.get('cuda_rng_states'):
            torch.cuda.set_rng_state_all(saved['cuda_rng_states'])
        if 'numpy_rng_state' in saved: np.random.set_state(saved['numpy_rng_state'])
        if 'python_rng_state' in saved: random.setstate(saved['python_rng_state'])
        print('RESUMING',model_name,protocol,fold,'at completed epoch',start_epoch)
    xt=torch.as_tensor(features_train,dtype=torch.float32)
    qtr=torch.as_tensor(Q[tr],dtype=torch.float32)
    avtr=torch.as_tensor(AV[tr],dtype=torch.float32)
    yt=torch.as_tensor(y[tr],dtype=torch.float32)
    ds=TensorDataset(xt,qtr,avtr,yt)
    for epoch in range(start_epoch,EPOCHS):
        generator=torch.Generator().manual_seed(RANDOM_SEED+10000+epoch)
        loader=DataLoader(ds,batch_size=NN_BATCH_SIZE,shuffle=True,generator=generator,num_workers=0)
        model.train();total_loss=0.0;seen=0
        for xb,qb,ab,yb in loader:
            xb,qb,ab,yb=xb.to(DEVICE),qb.to(DEVICE),ab.to(DEVICE),yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            yh=model(xb,qb,ab)
            loss=nn.functional.smooth_l1_loss(yh,yb,beta=1.)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()
            total_loss+=float(loss.detach().item())*len(yb);seen+=len(yb)
        losses.append(total_loss/max(1,seen))
        # Every epoch is atomically checkpointed, including optimizer for exact continuity.
        payload={'model_state':model.cpu().state_dict(),
                 'optimizer_state':optimizer.state_dict(),
                 'epoch':epoch+1,'losses':losses,'sha':DATA_SHA,'run':RUN_NAME,
                 'architecture':model_name,'seed':RANDOM_SEED,
                 'torch_rng_state':torch.get_rng_state(),
                 'cuda_rng_states':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                 'numpy_rng_state':np.random.get_state(),
                 'python_rng_state':random.getstate()}
        model.to(DEVICE)
        tmp=Path(str(ckpt)+'.tmp');torch.save(payload,tmp);os.replace(tmp,ckpt)
        append_row(LEARNING_FILE,{'protocol':protocol,'fold':fold,'model':model_name,
                          'step':epoch+1,'train_loss':losses[-1],'train_mae':np.nan})
        if (epoch+1)%10==0 or epoch==0 or epoch+1==EPOCHS:
            print(f'  {model_name} {protocol}/{fold}: epoch {epoch+1}/{EPOCHS}, Huber={losses[-1]:.4f}')
        if _REMOTE_AUTH_OK and ((epoch+1)%REMOTE_SYNC_EVERY_N_EPOCHS==0):
            remote_sync(why=f'{model_name} {protocol}/{fold} epoch {epoch+1}')
    model.eval()
    with torch.no_grad():
        xv=torch.as_tensor(features_test,dtype=torch.float32,device=DEVICE)
        qv=torch.as_tensor(Q[te],dtype=torch.float32,device=DEVICE)
        av=torch.as_tensor(AV[te],dtype=torch.float32,device=DEVICE)
        out=[]
        for i in range(0,len(xv),256):out.append(model(xv[i:i+256],qv[i:i+256],av[i:i+256]).cpu().numpy())
    return np.concatenate(out)

## 9. Run all predeclared model comparisons

This is the **main unattended execution cell**. It trains and checkpoints each model/fold, appends its metrics immediately on completion, and can continue after a crash if the checkpoint snapshot survives or is restored. It does not optimize models based on held-out 2025 metrics.

In [ ]:
# Final evaluation is run ONCE per model after predeclared random/spatial folds.
PROTOCOLS=OrderedDict([('random_diagnostic',random_splits),
                       ('spatial_primary',spatial_splits),
                       ('forward_2024_to_2025',forward_splits)])

def execute_job(protocol,fold,model_name,tr,te):
    if recorded(protocol,fold,model_name):
        print('SKIP complete:',protocol,fold,model_name);return
    start=time.monotonic()
    p,xtr,xte=get_inputs(tr,te)
    if model_name.startswith('MLP_') or model_name=='HydroSalNet_Gated':
        yp=fit_neural(model_name,protocol,fold,tr,te,p,xtr,xte)
    else:
        yp=fit_tree_or_linear(model_name,protocol,fold,tr,te,p,xtr,xte)
    assert len(yp)==len(te) and np.isfinite(yp).all(),model_name+' made invalid predictions'
    info=accuracy(y[te],yp)
    row={'protocol':protocol,'fold':fold,'model':model_name,'n_train':len(tr),'n_test':len(te),
         **info,'data_sha':DATA_SHA}
    preds=pd.DataFrame({'protocol':protocol,'fold':fold,'model':model_name,
                        'sample_id':df.iloc[te].sample_id.values,
                        'year':df.iloc[te].sample_year.values,'truth':y[te],'predicted':yp})
    # Transactional ordering: save model preprocessor and predictions BEFORE completion ledger.
    pre_path=WORK_DIR/'models'/f'{protocol}__{fold}__{model_name}__preprocessor.joblib'
    save_joblib_atomic({'preprocessor':p,'features':FEATURES,'groups':GROUPS,
                        'data_sha':DATA_SHA,'train_sample_ids':df.iloc[tr].sample_id.tolist()},pre_path)
    upsert_prediction_rows(PRED_FILE,preds)
    # The metrics ledger is the final commit marker for this fold/model.
    append_row_unique(METRIC_FILE,row,['protocol','fold','model'])
    print('COMPLETE',protocol,fold,model_name,'MAE',round(info['mae'],4),
          'RMSE',round(info['rmse'],4),'R2',round(info['r2'],4),'sec',round(time.monotonic()-start,1))
    if _REMOTE_AUTH_OK:remote_sync(why=f'Completed {model_name} {protocol}/{fold}')

for protocol,splits in PROTOCOLS.items():
    print('\n'+('='*72)+'\nPROTOCOL:',protocol,'|',len(splits),'folds\n'+('='*72))
    for tr,te,fold in splits:
        for model_name in MODEL_NAMES:
            execute_job(protocol,fold,model_name,tr,te)

print('\nALL MODEL TRAINING COMPLETED. Metrics:',METRIC_FILE)
remote_sync(force=True,why='All core models completed')

## 10. Model results, charts and diagnostics

Compare **MAE, RMSE, R² and signed bias**. Cross-fold averages are descriptive; a proper journal report should also include block uncertainty and the number of independent geographic groups. 2025 is a *final* test, not a model-tuning target.

In [ ]:
res=pd.read_csv(METRIC_FILE)
assert not res.empty,"Run training first"
comparison=res.groupby(['protocol','model'],as_index=False).agg(
    n_folds=('fold','count'),mae=('mae','mean'),rmse=('rmse','mean'),
    r2=('r2','mean'),bias=('bias','mean'),mae_std=('mae','std'))
comparison=comparison.sort_values(['protocol','mae'])
save_csv_atomic(comparison,WORK_DIR/'results'/'model_comparison_summary.csv')
for proto in PROTOCOLS:
    print('\n',proto)
    display(comparison.query('protocol == @proto')[['model','n_folds','mae','rmse','r2','mae_std']].round(4))

fig,axs=plt.subplots(1,2,figsize=(16,6))
sns.barplot(data=comparison,x='model',y='mae',hue='protocol',ax=axs[0])
axs[0].tick_params(axis='x',rotation=75);axs[0].set_title('MAE by model and evaluation protocol')
sns.barplot(data=comparison,x='model',y='r2',hue='protocol',ax=axs[1])
axs[1].tick_params(axis='x',rotation=75);axs[1].set_title('R² (negative values are possible)')
savefig('results_01_model_mae_r2')

fig,ax=plt.subplots(figsize=(13,5))
sns.barplot(data=comparison,x='model',y='rmse',hue='protocol',ax=ax)
ax.tick_params(axis='x',rotation=65);ax.set_title('RMSE comparison (mS/cm)')
savefig('results_02_model_rmse')

In [ ]:
# Held-out 2025 predictions, error analysis; do NOT use this display to retune models.
preds=pd.read_csv(PRED_FILE)
forward=preds.query("protocol == 'forward_2024_to_2025'")
if not forward.empty:
    fig,ax=plt.subplots(1,2,figsize=(13,5.1))
    chosen=['RandomForest','XGBoost','CatBoost','HydroSalNet_Gated']
    for m in chosen:
        g=forward[forward.model==m]
        ax[0].scatter(g.truth,g.predicted,s=16,alpha=.55,label=m)
    lo,hi=float(min(forward.truth.min(),forward.predicted.min())),float(max(forward.truth.max(),forward.predicted.max()))
    ax[0].plot([lo,hi],[lo,hi],'k--',lw=1,label='Perfect fit')
    ax[0].set_xlabel('Measured EC1:5');ax[0].set_ylabel('Predicted EC1:5');ax[0].legend(fontsize=8)
    ax[0].set_title('2025 sealed holdout: predictions')
    for m in chosen:
        g=forward[forward.model==m]
        ax[1].hist((g.predicted-g.truth).to_numpy(),bins=22,density=True,histtype='step',linewidth=1.6,label=m)
    ax[1].axvline(0,color='black',lw=1,linestyle='--');ax[1].legend(fontsize=8)
    ax[1].set_xlabel('Prediction − measured EC (mS/cm)');ax[1].set_title('2025 signed residual distribution')
    savefig('results_03_forward_holdout_predictions')

logs=pd.read_csv(LEARNING_FILE)
if not logs.empty:
    selected=logs[(logs.protocol=='forward_2024_to_2025')&logs.model.isin(['MLP_NoQA','MLP_EarlyFusion_QA','HydroSalNet_Gated'])]
    if not selected.empty:
        fig,ax=plt.subplots(figsize=(9,4.7))
        for m,g in selected.groupby('model'):
            g=g.groupby('step',as_index=False).train_loss.mean()
            ax.plot(g.step,g.train_loss,label=m)
        ax.set_title('Training Huber loss by neural epoch (not held-out score)')
        ax.set_xlabel('Epoch');ax.set_ylabel('Mean training Huber loss');ax.legend()
        savefig('results_04_neural_learning_curves')

## 11. Feature-group ablation study (spatial 2024 CV only)

Ablations compare Optical alone, Optical+SAR, Optical+Soil, Optical+Climate, all sources, and all sources with explicit group quality/missingness. Uses one **Random Forest (50 trees)** for controlled comparability. Missingness rates differ by source, so do not infer causality from an improved MAE.

In [ ]:
ABLS=OrderedDict({
    'Optical only':['S2_optical','L30_optical','S30_optical'],
    'Optical + SAR':['S2_optical','L30_optical','S30_optical','SAR'],
    'Optical + Soil':['S2_optical','L30_optical','S30_optical','Soil'],
    'Optical + Climate':['S2_optical','L30_optical','S30_optical','Climate'],
    'Climate + Soil + Hydro':['Climate','Soil','Hydrology','Terrain_landcover'],
    'All environmental groups':list(GROUPS.keys()),
    'All + quality':list(GROUPS.keys())
})
ABL_FILE=WORK_DIR/'results'/'ablation_spatial_cv.csv'
if RUN_ABLATIONS:
    if ABL_FILE.exists(): abls=pd.read_csv(ABL_FILE)
    else: abls=pd.DataFrame(columns=['experiment','fold','mae','rmse','r2'])
    for experiment,which in ABLS.items():
        cols=[FEATURES.index(c) for g in which for c in GROUPS[g]]
        for tr,te,fold in spatial_splits:
            if ((abls.experiment==experiment)&(abls.fold==fold)).any():continue
            raw_tr=Xraw[tr][:,cols];raw_te=Xraw[te][:,cols]
            pre=TrainOnlyPreprocessor().fit(raw_tr)
            a,b=pre.transform(raw_tr),pre.transform(raw_te)
            if experiment=='All + quality':
                a=augment(a,tr);b=augment(b,te)
            mdl=RandomForestRegressor(n_estimators=TREE_COUNT,min_samples_leaf=3,
                        max_features=.8,random_state=RANDOM_SEED,n_jobs=2)
            mdl.fit(a,y[tr]);metric=accuracy(y[te],mdl.predict(b))
            abls=pd.concat([abls,pd.DataFrame([{'experiment':experiment,'fold':fold,**metric}])],ignore_index=True)
            save_csv_atomic(abls,ABL_FILE)
    tbl=abls.groupby('experiment').agg(mae=('mae','mean'),rmse=('rmse','mean'),r2=('r2','mean')).sort_values('mae')
    display(tbl.round(4))
    fig,ax=plt.subplots(figsize=(10,5))
    ab_plot=abls.groupby('experiment').mae.agg(['mean','std']).sort_values('mean')
    ab_plot['mean'].plot.bar(ax=ax,yerr=ab_plot['std'].fillna(0),capsize=3)
    ax.tick_params(axis='x',rotation=50);ax.set_title('Ablation: 2024 spatial cross-validation (Random Forest)')
    savefig('results_05_group_ablation')
    remote_sync(why='Ablation results saved')

## 12. Save reusable model weights and final research outputs

Model checkpoints are stored separately for every model, protocol and fold. Each checkpoint has its **own fitted preprocessing object**, selected feature names, and the training sample IDs. For neural models `*.pt` saves model + optimizer + epoch, not just weights.

`hydrosalnet-recovery.zip` contains *models, checkpoints, metrics and manifests* but **not the original labeled CSV**. For a truly persistent record, confirm a Kaggle Notebook version was saved and/or the **private checkpoint Dataset** shows an updated version. Kaggle will not automatically save a newly edited `.ipynb` file from Python; you must use **Save Version** in the UI.

In [ ]:
# Save a compact human-readable run report and a recovery snapshot.
res=pd.read_csv(METRIC_FILE)
lines=[f'# HydroSalNet run: {RUN_NAME}', '',
       f'- Input SHA256: `{DATA_SHA}`',f'- Original master shape: {df.shape}',
       f'- Environmental predictor count: {len(FEATURES)}',
       f'- CV folds: {CV_FOLDS}',f'- Neural epochs: {EPOCHS}',
       f'- Tree/boosting budget: {BOOSTING_ROUNDS}',
       '- 2025 test must remain excluded from model selection.',
       '- 2024/2025 lab EC measurement protocol requires documentary verification.', '',
       '## Score summary (lower MAE is better)', '']
mean_table=res.groupby(['protocol','model'],as_index=False)[['mae','rmse','r2']].mean().sort_values(['protocol','mae'])
for row in mean_table.itertuples(index=False):
    lines.append(f'- {row.protocol} / {row.model}: MAE={row.mae:.4f}, RMSE={row.rmse:.4f}, R2={row.r2:.4f}')
(WORK_DIR/'run_summary.md').write_text('\n'.join(lines),encoding='utf-8')
zip_path=make_recovery_zip()
print('Training complete:')
print('  Model weights/checkpoints:',WORK_DIR/'checkpoints')
print('  Fitted preprocessing files:',WORK_DIR/'models')
print('  Score tables and plots:',WORK_DIR/'results')
print('  EDA figures:',WORK_DIR/'eda')
print('  Resumable snapshot:',zip_path,'(',round(zip_path.stat().st_size/1024**2,2),'MB)')
if _REMOTE_AUTH_OK:
    remote_sync(force=True,why='Final checkpoint bundle, model weights and score tables')
else:
    print('IMPORTANT: Private remote backup disabled. Save a completed Kaggle Notebook version;')
    print('otherwise unfinished session checkpoint files are not guaranteed to survive a crash.')

## 13. Reload a saved model for future predictions

This helper demonstrates **safe reuse of the trained model with its matching training-only preprocessing and exact feature order**. The example loads the forward-transfer model (trained only on 2024) and runs it on three 2025 records as a loading test; it is not a new performance estimate or a salinity advisory.

Load only checkpoints you created or from your trusted private Kaggle checkpoint Dataset.

In [ ]:
# Reload saved artifacts and predict consistently.
def load_and_predict(model_name, new_frame,
                     protocol='forward_2024_to_2025',fold='forward_2024_to_2025'):
    prefix=f'{protocol}__{fold}__{model_name}'
    bundle=joblib.load(WORK_DIR/'models'/(prefix+'__preprocessor.joblib'))
    assert bundle['data_sha']==DATA_SHA,'Different source dataset fingerprint'
    absent=[col for col in FEATURES if col not in new_frame.columns]
    if absent:raise ValueError('New frame must contain selected predictors (NaN permitted): '+str(absent))
    xframe=new_frame[FEATURES].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    xx=bundle['preprocessor'].transform(xframe.to_numpy(dtype=np.float32))
    ck=WORK_DIR/'checkpoints'/prefix
    if model_name in ('DummyMean','Ridge','ElasticNet','RandomForest','ExtraTrees'):
        mdl=joblib.load(str(ck)+'.joblib')['model'];return np.asarray(mdl.predict(xx))
    if model_name=='XGBoost':
        mdl=xgb.Booster(model_file=str(ck)+'.json');return mdl.predict(xgb.DMatrix(xx,feature_names=FEATURES))
    if model_name=='CatBoost':
        mdl=cb.CatBoostRegressor();mdl.load_model(str(ck)+'.cbm');return np.asarray(mdl.predict(xx)).reshape(-1)
    if model_name=='LightGBM':
        mdl=lgb.Booster(model_file=str(ck)+'.txt');return np.asarray(mdl.predict(xx))
    if model_name in ('MLP_NoQA','MLP_EarlyFusion_QA','HydroSalNet_Gated'):
        qn,an=quality_arrays(new_frame)
        xn=np.concatenate([xx,qn,an],axis=1) if model_name=='MLP_EarlyFusion_QA' else xx
        net=QualityAwareGatedNet() if model_name=='HydroSalNet_Gated' else SmallMLP(xn.shape[1])
        state=torch.load(str(ck)+'.pt',map_location='cpu',weights_only=False)
        net.load_state_dict(state['model_state']);net.eval()
        with torch.no_grad():
            pred=net(torch.tensor(xn,dtype=torch.float32),
                     torch.tensor(qn,dtype=torch.float32),torch.tensor(an,dtype=torch.float32))
        return pred.numpy()
    raise ValueError('Unknown model '+model_name)

example=df[df.sample_year==2025].head(3).copy()
example['reloaded_RF_prediction']=load_and_predict('RandomForest',example)
example['reloaded_Gated_prediction']=load_and_predict('HydroSalNet_Gated',example)
display(example[['sample_id','reloaded_RF_prediction','reloaded_Gated_prediction']])
save_csv_atomic(example[['sample_id','reloaded_RF_prediction','reloaded_Gated_prediction']],
                WORK_DIR/'results'/'example_reloaded_model_predictions.csv')

## 14. Resume after a crash / reopen later

**Always restart from the FIRST configuration cell** (or choose **Run All**). No model-training cell can work in a fresh kernel if earlier cells were not run; the common `NameError: WORK_DIR not defined` is caused by out-of-order execution. The notebook now keeps all initialization earlier and displays the correct detected data path.

### If Kaggle says FileNotFound
- The dataset's *display title* is not necessarily its CSV filename or mount directory. This notebook scans `/kaggle/input` recursively and shows all CSV paths. **Attach HydroSalNet using Add Input**. If it still fails, inspect the printed files; upload the final master CSV itself, not just an image or notebook.
- This release verifies `349 x 548` and checks `sample_id`, target EC and campaign metadata. It does **not** silently select a smaller older master.

### If your computer shuts down or you go to sleep
- **Save Version → Save & Run All** runs in Kaggle's cloud; closing the laptop or losing *your* Internet connection does not by itself stop that run. A Kaggle session limit, remote environment failure, or user cancellation **can** stop it.
- `/kaggle/working/<run>/` contains local weights, optimizer states, preprocessing, metrics and EDA charts. These are persisted in the outputs of a **completed and successfully saved notebook version**, not necessarily an interrupted one. Use the optional verified private Kaggle Dataset sync for mid-run recovery.
- For private remote sync: set `ENABLE_REMOTE_BACKUP=True`, turn Kaggle Internet ON, create secrets `KAGGLE_USERNAME`/`KAGGLE_KEY`, and verify **`[REMOTE BACKUP PREFLIGHT VERIFIED]`** before unattended training. Default is OFF to avoid unauthorized account actions.

### Recovery
1. Attach the **same** final master CSV and (if available) your **private** `hydrosalnet-recovery.zip` from the latest backup as **Kaggle Inputs**.
2. Use the **same** `RUN_NAME` and all the same feature/training settings; never reassign outputs to `/kaggle/input` (read-only).
3. Click **Run All**. Already completed fold/model pairs are skipped, Random Forest and boosting models continue from the last durable tree chunk, and neural networks continue from the latest saved epoch plus optimizer state.
4. If you intentionally switch from 10 epochs to 50, or alter features/hyperparameters, use **a new RUN_NAME**, otherwise the ledger could represent an incompatible experiment; this release stops such mixed resumes.

**Generated files**: `results/all_fold_metrics.csv`, `results/model_comparison_summary.csv`, `results/out_of_fold_predictions.csv`, `results/ablation_spatial_cv.csv`, `logs/learning_history.csv`, `checkpoints/*`, `models/*__preprocessor.joblib`, `eda/*.png`, `hydrosalnet-recovery.zip`, `run_summary.md`, `manifest.json`, and frozen `splits_frozen.csv`.

**Academic disclaimer:** Ground EC1:5 is **not** saturated-paste ECe. Cross-campaign transfer has mixed year/season/site shifts. Spatial 4-fold KMeans blocks are pragmatic exploratory groups, not a full spatial-autocorrelation-aware holdout. Before publication, verify both campaigns' laboratory method, ground provenance, repeated-site independence, block size sensitivity and external validation.
